# How Transformers Work — Visual Walkthrough
This notebook illustrates tokenization, embeddings, and attention (Q, K, V) with real code and visualizations.

---
## Part 1 — Tokenization
A tokenizer splits text into sub-word pieces and maps each to an integer ID.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sentence = "The cat sat on the mat"
tokens   = tokenizer.tokenize(sentence)
ids      = tokenizer.convert_tokens_to_ids(tokens)

print(f"Sentence : {sentence!r}")
print(f"Tokens   : {tokens}")
print(f"Token IDs: {ids}")
print(f"\nSequence length = {len(tokens)} tokens")
print("\n'4k context window' means the model can process 4096 tokens at once.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 1.8))
ax.set_xlim(0, len(tokens))
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title("Token IDs for each word", fontsize=13)

colors = plt.cm.Pastel1.colors
for i, (tok, tid) in enumerate(zip(tokens, ids)):
    rect = mpatches.FancyBboxPatch(
        (i + 0.05, 0.2), 0.85, 0.6,
        boxstyle="round,pad=0.05", linewidth=1.2,
        edgecolor='#555', facecolor=colors[i % len(colors)]
    )
    ax.add_patch(rect)
    ax.text(i + 0.5, 0.63, tok,   ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(i + 0.5, 0.33, str(tid), ha='center', va='center', fontsize=9, color='#333')

plt.tight_layout()
plt.show()

---
## Part 2 — Embeddings
Each token ID is looked up in an **embedding table** and converted to a dense vector of `hidden_dim` numbers.

With BERT's `hidden_dim = 768`, our 6-token sentence becomes a **matrix of shape (6, 768)**.

In [ ]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

input_ids = torch.tensor([tokenizer.encode(sentence, add_special_tokens=False)])

# Pull just the raw token embeddings (before attention layers)
with torch.no_grad():
    raw_embeds = model.embeddings.word_embeddings(input_ids)  # shape: (1, seq_len, 768)

embeds = raw_embeds[0]  # (seq_len, hidden_dim)
print(f"Input shape : {input_ids.shape}  → (batch=1, seq_len={input_ids.shape[1]})")
print(f"Embedding shape: {embeds.shape}  → (seq_len, hidden_dim)")
print(f"\n'cat' token embedding (first 10 of 768 numbers):")
print(embeds[1, :10].numpy().round(4))

In [ ]:
import numpy as np
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: heatmap of the full embedding matrix
ax = axes[0]
data = embeds.numpy()  # (6, 768)
# Show first 64 dims for readability
sns.heatmap(data[:, :64], ax=ax, cmap='RdBu_r', center=0,
            xticklabels=False, yticklabels=tokens, cbar_kws={'shrink': 0.7})
ax.set_title(f"Embedding matrix: shape {list(embeds.shape)}\n(showing first 64 of 768 dims)", fontsize=11)
ax.set_xlabel("Hidden dimension →")
ax.set_ylabel("Token")

# Right: cosine similarity between token embeddings
ax2 = axes[1]
norms = data / (np.linalg.norm(data, axis=1, keepdims=True) + 1e-8)
sim = norms @ norms.T
sns.heatmap(sim, ax=ax2, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=tokens, yticklabels=tokens, vmin=0, vmax=1)
ax2.set_title("Cosine similarity between\nraw token embeddings", fontsize=11)

plt.tight_layout()
plt.show()

---
## Part 3 — Attention: Query, Key, Value

For each token, the model produces three vectors:
- **Q (Query)** — "what am I looking for?"
- **K (Key)**   — "what do I contain?"
- **V (Value)** — "here's my actual content"

```
Q = input @ Wq
K = input @ Wk
V = input @ Wv

attention_scores = softmax(Q @ K.T / sqrt(head_dim))
output           = attention_scores @ V
```

In [ ]:
import math

# ── Manual single-head attention (no batching, no model weights — pure math) ──
seq_len, hidden_dim = embeds.shape   # (6, 768)
head_dim = 64  # smaller head dim for illustration

torch.manual_seed(42)
Wq = torch.randn(hidden_dim, head_dim) * 0.02
Wk = torch.randn(hidden_dim, head_dim) * 0.02
Wv = torch.randn(hidden_dim, head_dim) * 0.02

Q = embeds @ Wq   # (6, 64)
K = embeds @ Wk   # (6, 64)
V = embeds @ Wv   # (6, 64)

print(f"Input  shape: {list(embeds.shape)}")
print(f"Q shape:      {list(Q.shape)}   (seq_len, head_dim)")
print(f"K shape:      {list(K.shape)}")
print(f"V shape:      {list(V.shape)}")

# Attention scores
scores = Q @ K.T / math.sqrt(head_dim)        # (6, 6)
attn   = torch.softmax(scores, dim=-1)         # (6, 6)  — rows sum to 1
output = attn @ V                              # (6, 64)

print(f"\nAttention weights shape: {list(attn.shape)}  (each row = how much token i attends to each j)")
print(f"Output shape:            {list(output.shape)}")

In [ ]:
# Visualise the raw attention weight matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
attn_np = attn.detach().numpy()
sns.heatmap(attn_np, ax=ax, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=tokens, yticklabels=tokens, vmin=0, vmax=1)
ax.set_title("Attention weights\n(row i → how much token i attends to each j)", fontsize=11)
ax.set_xlabel("Key tokens  (what I pay attention to)")
ax.set_ylabel("Query token (who is asking)")

# Q·K dot products before softmax
ax2 = axes[1]
sns.heatmap(scores.detach().numpy(), ax=ax2, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=tokens, yticklabels=tokens)
ax2.set_title("Raw Q·Kᵀ scores (before softmax)\n÷ √head_dim to prevent vanishing gradients", fontsize=11)
ax2.set_xlabel("Key tokens")
ax2.set_ylabel("Query token")

plt.tight_layout()
plt.show()

---
## Part 4 — Real Attention from BERT (learned weights)

BERT has **12 attention heads** per layer, each learning different relationships.  
Let's extract and visualise the actual learned attention from layer 0.

In [ ]:
# Run the full model with output_attentions=True
input_ids_with_special = tokenizer.encode(sentence, return_tensors='pt')  # includes [CLS], [SEP]
tokens_with_special    = tokenizer.convert_ids_to_tokens(input_ids_with_special[0])

with torch.no_grad():
    outputs = model(input_ids_with_special, output_attentions=True)

# outputs.attentions: tuple of (num_layers=12,) each shape (batch, heads, seq, seq)
layer0_attn = outputs.attentions[0][0]  # (12 heads, seq_len, seq_len)
print(f"Tokens: {tokens_with_special}")
print(f"Layer 0 attention shape: {list(layer0_attn.shape)}  (heads, seq, seq)")

In [ ]:
# Show all 12 heads of layer 0
num_heads = layer0_attn.shape[0]
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle("BERT Layer 0 — All 12 Attention Heads\n"
             "Each head learns different token relationships", fontsize=13, y=1.01)

for head_idx, ax in enumerate(axes.flat):
    head_attn = layer0_attn[head_idx].numpy()  # (seq, seq)
    sns.heatmap(head_attn, ax=ax, cmap='Blues', vmin=0, vmax=1,
                xticklabels=tokens_with_special,
                yticklabels=tokens_with_special,
                cbar=False, linewidths=0.3)
    ax.set_title(f"Head {head_idx}", fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.show()

---
## Part 5 — The "it" → "cat" example
Use a sentence with a pronoun to see whether attention recovers the coreference.

In [ ]:
coref_sentence = "The cat sat because it was tired"
coref_ids   = tokenizer.encode(coref_sentence, return_tensors='pt')
coref_toks  = tokenizer.convert_ids_to_tokens(coref_ids[0])

with torch.no_grad():
    coref_out = model(coref_ids, output_attentions=True)

# Try several layers; later layers tend to capture semantics better
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f"'The cat sat because it was tired'\n"
              "Does any head learn that 'it' → 'cat'?", fontsize=13)

it_idx = coref_toks.index('it')

for ax, (layer, head) in zip(axes.flat, [(0,0),(0,4),(3,2),(5,6),(8,3),(11,10)]):
    attn_mat = coref_out.attentions[layer][0][head].numpy()
    sns.heatmap(attn_mat, ax=ax, cmap='Oranges', vmin=0, vmax=attn_mat.max(),
                xticklabels=coref_toks, yticklabels=coref_toks,
                cbar=False, linewidths=0.3)
    # Highlight the 'it' row
    ax.add_patch(plt.Rectangle((0, it_idx), len(coref_toks), 1,
                                fill=False, edgecolor='red', lw=2))
    ax.set_title(f"Layer {layer}, Head {head}", fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

plt.tight_layout()
plt.show()
print("Red box = 'it' row. Look for a bright cell under 'cat' — that head learned coreference.")

---
## Summary

| Concept | Shape | What it is |
|---|---|---|
| Token IDs | `(seq_len,)` | Integer index into vocabulary |
| Embedding matrix | `(seq_len, hidden_dim)` | Each token → dense vector |
| Q, K, V | `(seq_len, head_dim)` | Per-token search/content vectors |
| Attention weights | `(seq_len, seq_len)` | How much each token attends to each other |
| Attention output | `(seq_len, head_dim)` | Context-aware representation |

- **Context window** = maximum `seq_len` the model handles (4k, 128k, etc.)
- **hidden_dim** = width of the embedding (768 for BERT-base, 4096 for larger models)
- **Wq, Wk, Wv** are learned matrices — the actual parameters trained via gradient descent